# Validation and model selection

In [Overfitting and the train/test split](overfitting_and_the_split.ipynb), the depth of a tree was chosen by looking at its test error, with a warning: once the test set has been used to make a decision, its error stops being an honest estimate. Choosing *between models* is the same kind of decision, and the most common one: a linear model or a forest, ten neighbours or fifty. Every choice made by looking at the test set makes its error look better than the error the chosen model will have on new data.

## Choosing on the test set

How much better, and how often? An experiment on California housing: a small sample of 600 districts, 400 to train and 200 to test, and 45 candidate models (trees of depth 1 to 15, and *k*-nearest neighbours with *k* from 1 to 30). The candidate with the lowest test error is chosen, and then measured again on the 20,000 districts that played no part, which stand for the new data the model will meet. The whole experiment is repeated with 30 different samples.

In [ ]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True).frame
X = housing.drop(columns="MedHouseVal")
y = housing["MedHouseVal"]

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

candidates = (
    [DecisionTreeRegressor(max_depth=depth, random_state=0) for depth in range(1, 16)]
    + [make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=k)) for k in range(1, 31)]
)
len(candidates)

45

`make_pipeline` joins the scaler and the neighbours into one model, so that each candidate is fitted and used in a single call; [California housing with pipelines](../basic-workflows/california_with_pipelines.ipynb) explains it. The experiment:

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split


def choose_on_the_test_set(seed):
    X_sample, X_fresh, y_sample, y_fresh = train_test_split(X, y, train_size=600, random_state=seed)
    X_train, X_test, y_train, y_test = train_test_split(X_sample, y_sample, test_size=200, random_state=seed)
    test_errors, fresh_errors = [], []
    for candidate in candidates:
        candidate.fit(X_train, y_train)
        test_errors.append(mean_absolute_error(y_test, candidate.predict(X_test)))
        fresh_errors.append(mean_absolute_error(y_fresh, candidate.predict(X_fresh)))
    chosen = np.argmin(test_errors)
    return {"test MAE of the chosen model": test_errors[chosen], "its MAE on fresh data": fresh_errors[chosen]}


runs = 100_000 * pd.DataFrame([choose_on_the_test_set(seed) for seed in range(30)])
runs["optimism"] = runs["its MAE on fresh data"] - runs["test MAE of the chosen model"]
runs.describe().loc[["mean", "min", "50%", "max"]].round(0)

,test MAE of the chosen model,its MAE on fresh data,optimism
mean,51982.0,54166.0,2183.0
min,46375.0,51302.0,-2360.0
50%,52454.0,53582.0,1581.0
max,56448.0,60535.0,7222.0


In [ ]:
(runs["optimism"] > 0).mean()

np.float64(0.7333333333333333)

On average, the chosen model's test error was about \$2,200 lower than its error on fresh data, and it was lower in about three runs out of four. No candidate learned from the test rows: the bias comes from the choice alone. Among 45 candidates, the one that looks best on 200 particular districts is partly the one that happened to suit those districts. The more candidates are compared, and the fewer the test rows, the larger the effect. In the runs where the difference is negative, chance went the other way: the estimate is optimistic on average, not in every run.

## Three sets

The way out is to keep the decisions away from the test set. The data is split three ways:

| Set | Used for | How often |
| --- | --- | --- |
| **Training** | fitting each candidate's parameters | every time a candidate is trained |
| **Validation** | comparing candidates and choosing settings | as often as needed |
| **Test** | measuring the chosen model | once, at the end |

The validation set takes over the role the test set played in the experiment above, and it ends up just as optimistic, which is fine: its job is to rank candidates, not to report a number. The test set is left untouched until every decision has been made, so its error remains an honest estimate.

`train_test_split` splits in two, so it is called twice: first the test set is set aside, then the rest is split into training and validation. A quarter of the remaining 80% is 20% of the whole:

In [ ]:
X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_validation, y_train, y_validation = train_test_split(X_rest, y_rest, test_size=0.25, random_state=42)
len(X_train), len(X_validation), len(X_test)

(12384, 4128, 4128)

## Comparing models on the validation set

Four candidates of different kinds, each trained on the training set and measured on the validation set. The scaler goes into the pipeline of the neighbours, which depend on distances, and into the linear model's, where it changes no prediction but makes the weights comparable; it is fitted on the training rows only, as in [California housing, step by step](../basic-workflows/california_by_hand.ipynb):

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

models = {
    "linear regression": make_pipeline(StandardScaler(), LinearRegression()),
    "k-nearest neighbours": make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=10)),
    "decision tree": DecisionTreeRegressor(max_depth=10, random_state=42),
    "random forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
}
for model in models.values():
    model.fit(X_train, y_train)

In [ ]:
validation_errors = pd.DataFrame({
    name: {
        "training MAE ($)": 100_000 * mean_absolute_error(y_train, model.predict(X_train)),
        "validation MAE ($)": 100_000 * mean_absolute_error(y_validation, model.predict(X_validation)),
    }
    for name, model in models.items()
}).T.round(0)
validation_errors

,training MAE ($),validation MAE ($)
linear regression,52665.0,53333.0
k-nearest neighbours,39078.0,44462.0
decision tree,30039.0,42424.0
random forest,12415.0,33555.0


The [random forest](../algorithms/tree-based/random_forests.ipynb) has the lowest validation error, about \$33,600, against about \$53,300 for the linear model (the model of [California housing, step by step](../basic-workflows/california_by_hand.ipynb), with about the same error). It also has by far the largest gap between its training and validation errors: about \$12,400 against \$33,600. A gap is a sign of overfitting, but it doesn't disqualify a model: what decides is the error on data it didn't learn from, and there the forest is still the best of the four. Its depth and number of trees could be tuned on this validation set as well; [Cross-validation and hyperparameter search](cross_validation_and_search.ipynb) does it systematically.

In [ ]:
chosen_name = validation_errors["validation MAE ($)"].idxmin()
chosen_name

'random forest'

## Testing once

The choice is made, so the test set can be used, once. The chosen kind of model is trained again on the training and validation sets together: the validation rows have done their job, and more data usually gives a better model. Its test error is the estimate to report:

In [ ]:
from sklearn.base import clone

final_model = clone(models[chosen_name]).fit(X_rest, y_rest)
100_000 * mean_absolute_error(y_test, final_model.predict(X_test))

32773.10800872095

About \$32,800, close to the forest's validation error and slightly lower, partly because it has now learned from 16,512 districts instead of 12,384, partly because the test districts are other districts; and with only four candidates compared, the validation error had little room to be optimistic. This is the estimate to report for this model.

After this number is seen, nothing is changed. If it is disappointing and the model is changed in response, the test set has been used to make a decision, and the next test error would be optimistic in the same way as in the experiment above. An honest estimate would then need data that hasn't been used yet.

## The price of a validation set

A validation set is data the model doesn't learn from, and with little data that hurts twice: fewer rows to train on, and a validation error measured on few rows, which changes noticeably with which rows happen to fall into it. With 600 districts, as in the experiment, a 20% validation set would be 120 districts. [Cross-validation and hyperparameter search](cross_validation_and_search.ipynb) gets more out of the same rows by taking turns: each part of the data serves once as the validation set.